# Credit Card Default Prediction

Predict which credit-card clients will default next month, from their credit limit, demographics and six months of repayment history. The classes are imbalanced (about 22% default), so the notebook compares plain classifiers with imbalance-aware ones and judges them on recall, F1 and ROC AUC on a held-out test set, not on accuracy.

**Data:** *Default of Credit Card Clients* (Taiwan, 2005), 30,000 clients, from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients), CC BY 4.0. Yeh, I. C. and Lien, C. H. (2009), *Expert Systems with Applications* 36(2).

## Variables

- **LIMIT_BAL**: credit granted (NT dollars), individual plus family credit.
- **SEX** (1 = male, 2 = female), **EDUCATION** (1 = graduate school, 2 = university, 3 = high school, 4 = others), **MARRIAGE** (1 = married, 2 = single, 3 = others), **AGE**.
- **PAY_1 … PAY_6**: repayment status from September back to April 2005. -2 = no consumption, -1 = paid in full, 0 = revolving credit, 1-9 = months of payment delay.
- **BILL_AMT1 … BILL_AMT6**: bill statement amounts, same months.
- **PAY_AMT1 … PAY_AMT6**: amounts paid, same months.
- **TARGET**: default in the following month (1 = yes).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import metrics
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline as sk_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from imblearn.ensemble import BalancedBaggingClassifier, BalancedRandomForestClassifier
from imblearn.pipeline import make_pipeline
from imblearn.under_sampling import RandomUnderSampler

SEED = 2022
sns.set_theme(style="whitegrid")

## Helpers

In [ ]:
def performance(model, X_train, y_train, X_test, y_test):
    """Accuracy, precision, recall, F1 and ROC AUC on train and test.
    Every sklearn metric is called as metric(y_true, y_pred)."""
    rows = {}
    for split, X, y in [("Train", X_train, y_train), ("Test", X_test, y_test)]:
        pred = model.predict(X)
        proba = model.predict_proba(X)[:, 1]
        rows[split] = {
            "Accuracy": metrics.accuracy_score(y, pred),
            "Precision": metrics.precision_score(y, pred),
            "Recall": metrics.recall_score(y, pred),
            "F1": metrics.f1_score(y, pred),
            "ROC AUC": metrics.roc_auc_score(y, proba),
        }
    return pd.DataFrame(rows)


def plot_confusion(model, X_test, y_test, title):
    cm = metrics.confusion_matrix(y_test, model.predict(X_test))
    names = ["True neg", "False pos", "False neg", "True pos"]
    labels = [f"{n}\n{c:,}\n{c / cm.sum():.1%}" for n, c in zip(names, cm.flatten())]
    plt.figure(figsize=(4.5, 3.5))
    sns.heatmap(cm, annot=np.array(labels).reshape(2, 2), fmt="", cmap="Blues", cbar=False)
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.title(title)
    plt.show()


results = {}

def evaluate(name, model):
    model.fit(X_train, y_train)
    table = performance(model, X_train, y_train, X_test, y_test)
    results[name] = table["Test"]
    print(metrics.classification_report(y_test, model.predict(X_test), digits=3))
    plot_confusion(model, X_test, y_test, name)
    return table

## Loading the data

In [ ]:
# The first row of the sheet holds the real column names.
ds = pd.read_excel("data/default_credit_card_clients.xlsx", header=1)
ds = ds.drop(columns=["ID"]).rename(columns={"PAY_0": "PAY_1", "default payment next month": "TARGET"})
print(ds.shape)
print("duplicated rows:", ds.duplicated().sum(), "| missing values:", ds.isna().sum().sum())
ds.describe().T

## Exploration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(data=ds, x="LIMIT_BAL", kde=True, ax=axes[0])
sns.histplot(data=ds, x="AGE", kde=True, ax=axes[1])
sns.countplot(data=ds, x="TARGET", ax=axes[2])
plt.tight_layout()
plt.show()
print("LIMIT_BAL skew:", round(ds["LIMIT_BAL"].skew(), 2), "| AGE skew:", round(ds["AGE"].skew(), 2))
ds["TARGET"].value_counts(normalize=True)

Credit limits are right-skewed and about 22% of clients default, so accuracy alone would reward predicting "no default" for everyone.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, ax in zip(["SEX", "EDUCATION", "MARRIAGE"], axes):
    sns.countplot(data=ds, x=col, ax=ax)
plt.tight_layout()
plt.show()

EDUCATION has codes 0, 5 and 6 and MARRIAGE has code 0, none of which the data description defines. Those rows are dropped.

In [ ]:
ds = ds[~ds["EDUCATION"].isin([0, 5, 6]) & (ds["MARRIAGE"] != 0)]
ds.shape

In [ ]:
pay = [f"PAY_{i}" for i in range(1, 7)]
fig, axes = plt.subplots(2, 3, sharey=True, figsize=(15, 7))
for col, ax in zip(pay, axes.flatten()):
    sns.countplot(data=ds, x=col, ax=ax)
plt.tight_layout()
plt.show()

In the repayment-status columns, -2 means no consumption that month. It is a valid state, so those rows are kept.

In [ ]:
bill = [f"BILL_AMT{i}" for i in range(1, 7)]
pay_amt = [f"PAY_AMT{i}" for i in range(1, 7)]
plt.figure(figsize=(10, 8))
sns.heatmap(ds[["LIMIT_BAL", "AGE"] + bill + pay_amt].corr(), cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Pearson correlation of the numeric variables")
plt.show()

The six bill amounts are highly correlated with each other, so they are replaced by their mean. Payment amounts are heavily right-skewed, so they enter as `log1p`.

In [ ]:
for col in pay_amt:
    print(f"{col}: skew {ds[col].skew():6.2f} -> log1p skew {np.log1p(ds[col]).skew():5.2f}")

## Data preparation

In [ ]:
X = ds.copy()
X["BILL_AMT"] = X[bill].mean(axis=1)
for col in pay_amt:
    X[f"LOG_{col}"] = np.log1p(X[col])
X = X.drop(columns=bill + pay_amt)

y = X.pop("TARGET")
categorical = ["SEX", "EDUCATION", "MARRIAGE"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, stratify=y, random_state=SEED)

# One-hot encode the categorical columns, fitted on train only.
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(X_train[categorical])

def encode(frame):
    dummies = pd.DataFrame(encoder.transform(frame[categorical]),
                           columns=encoder.get_feature_names_out(categorical), index=frame.index)
    return pd.concat([frame.drop(columns=categorical), dummies], axis=1)

X_train, X_test = encode(X_train), encode(X_test)
print(X_train.shape, X_test.shape)
y_train.value_counts(normalize=True)

## Models
### Logistic regression (standardised features)

In [ ]:
evaluate("Logistic regression", sk_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)))

### Random forest

In [ ]:
evaluate("Random forest", RandomForestClassifier(max_depth=10, random_state=SEED, n_jobs=-1))

Both plain models predict "no default" for most clients and miss many defaulters. The next models correct for the imbalance.

### Balanced random forest
Each tree is grown on a bootstrap sample under-sampled to balance the classes.

In [ ]:
evaluate("Balanced random forest", BalancedRandomForestClassifier(
    max_depth=10, sampling_strategy="all", replacement=True, bootstrap=False, random_state=SEED, n_jobs=-1))

### Balanced bagging

In [ ]:
evaluate("Balanced bagging", BalancedBaggingClassifier(n_estimators=50, random_state=SEED, n_jobs=-1))

### Random under-sampling + random forest

In [ ]:
evaluate("Under-sampling + random forest", make_pipeline(
    RandomUnderSampler(random_state=SEED),
    RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_split=8, random_state=SEED, n_jobs=-1)))

### Random under-sampling + gradient boosting

In [ ]:
evaluate("Under-sampling + gradient boosting", make_pipeline(
    RandomUnderSampler(random_state=SEED),
    GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=SEED)))

## Comparison on the test set

In [ ]:
summary = pd.DataFrame(results).T.sort_values("F1", ascending=False)
summary.round(3)

In [ ]:
# Sanity checks: every model ran, and the best one ranks clients clearly better than chance.
assert len(summary) == 6
assert summary["ROC AUC"].max() > 0.7

## Conclusion

CONCLUSION_PLACEHOLDER